# Pertemuan 4 — Divide and Conquer I
Merge Sort, Quick Sort (pivot pertama vs acak), Binary Search, dan Quickselect. Dibandingkan secara **teoritis (hitung operasi)** dan **empiris (waktu)**.

In [ ]:
import random, sys, time, math, bisect
import numpy as np
import matplotlib.pyplot as plt
sys.setrecursionlimit(20000)
random.seed(7)

## 1. Implementasi dengan penghitung operasi

In [ ]:
def merge_sort(a, cnt):
    if len(a) <= 1: return a
    m = len(a) // 2
    L, R = merge_sort(a[:m], cnt), merge_sort(a[m:], cnt)
    out, i, j = [], 0, 0
    while i < len(L) and j < len(R):
        cnt[0] += 1
        if L[i] <= R[j]: out.append(L[i]); i += 1
        else: out.append(R[j]); j += 1
    return out + L[i:] + R[j:]

def quick_sort(a, cnt, pivot="first"):
    if len(a) <= 1: return a
    p = a[0] if pivot == "first" else random.choice(a)
    less = [x for x in a if x < p]; eq = [x for x in a if x == p]; more = [x for x in a if x > p]
    cnt[0] += len(a)                       # satu pass partisi ≈ n perbandingan
    return quick_sort(less, cnt, pivot) + eq + quick_sort(more, cnt, pivot)

def quickselect(a, k, cnt):                # k-th terkecil (0-based), rata-rata O(n)
    p = random.choice(a)
    less = [x for x in a if x < p]; eq = [x for x in a if x == p]; more = [x for x in a if x > p]
    cnt[0] += len(a)
    if k < len(less): return quickselect(less, k, cnt)
    if k < len(less) + len(eq): return p
    return quickselect(more, k - len(less) - len(eq), cnt)

def binary_search(a, x):
    lo, hi, steps = 0, len(a) - 1, 0
    while lo <= hi:
        steps += 1; mid = (lo + hi) // 2
        if a[mid] == x: return mid, steps
        lo, hi = (mid + 1, hi) if a[mid] < x else (lo, mid - 1)
    return -1, steps

a = random.sample(range(10000), 500)
c = [0]; assert merge_sort(a, c) == sorted(a)
c = [0]; assert quick_sort(a, c) == sorted(a)
c = [0]; assert quickselect(a, 250, c) == sorted(a)[250]
print("Semua algoritma benar ✔")

## 2. Pengaruh distribusi data (n = 2000)
Quick Sort dengan pivot pertama → **O(n²)** pada data terurut; pivot acak → O(n log n) *expected*.

In [ ]:
n = 2000
datasets = {
    "acak":        random.sample(range(10*n), n),
    "terurut":     list(range(n)),
    "terbalik":    list(range(n, 0, -1)),
    "banyak duplikat": [random.randint(0, 9) for _ in range(n)],
}
print(f"{'data':16s} {'merge':>9} {'quick-first':>12} {'quick-random':>13}   n·log2n={n*math.log2(n):.0f}, n²/2={n*n//2}")
for name, d in datasets.items():
    r = []
    for f in (lambda x, c: merge_sort(x, c), lambda x, c: quick_sort(x, c, "first"), lambda x, c: quick_sort(x, c, "random")):
        c = [0]; f(d, c); r.append(c[0])
    print(f"{name:16s} {r[0]:>9} {r[1]:>12} {r[2]:>13}")

## 3. Skala pertumbuhan (data acak) dan waktu nyata

In [ ]:
ns = [500, 1000, 2000, 4000, 8000, 16000]
tm, tq, tb = [], [], []
for n in ns:
    d = random.sample(range(10*n), n)
    for lst, f in ((tm, lambda x: merge_sort(x, [0])), (tq, lambda x: quick_sort(x, [0], "random")), (tb, sorted)):
        t = time.perf_counter(); f(d); lst.append(time.perf_counter() - t)
plt.loglog(ns, tm, "o-", label="merge sort"); plt.loglog(ns, tq, "s-", label="quick sort (acak)")
plt.loglog(ns, tb, "^-", label="sorted() Timsort (C)"); plt.legend(); plt.xlabel("n"); plt.ylabel("detik"); plt.grid(True, which="both"); plt.show()

## 4. Quickselect vs sort penuh (median)

In [ ]:
for n in [10**4, 10**5]:
    d = random.sample(range(10*n), n)
    c = [0]; t = time.perf_counter(); m = quickselect(d, n // 2, c); t1 = time.perf_counter() - t
    t = time.perf_counter(); m2 = sorted(d)[n // 2]; t2 = time.perf_counter() - t
    print(f"n={n}: quickselect ≈ {c[0]/n:.1f}·n operasi ({t1:.3f}s) | sorted ({t2:.3f}s) | sama? {m == m2}")

## 5. Latihan / Tugas
1. Ganti pivot menjadi **median-of-three**. Apakah data terurut masih O(n²)?
2. Tambahkan *cutoff*: untuk n ≤ 16 gunakan insertion sort. Cari cutoff terbaik secara empiris.
3. Buktikan secara empiris bahwa `binary_search` butuh ≤ ⌊log₂ n⌋+1 langkah (uji untuk semua elemen).
4. **Tugas:** laporan 1 halaman — prediksi teoritis vs hasil benchmark (tabel + grafik + penjelasan selisih).

In [ ]:
def median_of_three_quick_sort(a, cnt):
    pass  # TODO